In [1]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

# Inicializace kamery robota
camera = Camera.instance(
    width=224,
    height=224
)

# Widget pro zobrazení aktuálního snímku
image_widget = widgets.Image(
    format='jpeg',
    width=224,
    height=224
)

# Propojení kamery s widgetem (živý náhled)
camera_link = traitlets.dlink(
    (camera, 'value'),
    (image_widget, 'value'),
    transform=bgr8_to_jpeg
)

# Zobrazení v Jupyter Notebooku
display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [2]:
import os

DATASET_DIR = 'dataset_navigator_10class'

dirs = {
    '0_dead_end': os.path.join(DATASET_DIR, 'dead_end'),
    '1_corridor': os.path.join(DATASET_DIR, 'corridor'),
    '2_corner_left': os.path.join(DATASET_DIR, 'corner_left'),
    '3_corner_right': os.path.join(DATASET_DIR, 'corner_right'),
    '4_t_junction_lr': os.path.join(DATASET_DIR, 't_junction_lr'),
    '5_t_junction_sl': os.path.join(DATASET_DIR, 't_junction_sl'),
    '6_t_junction_sr': os.path.join(DATASET_DIR, 't_junction_sr'),
    '7_goal': os.path.join(DATASET_DIR, 'goal'),
    '8_drift_left': os.path.join(DATASET_DIR, 'drift_left'),
    '9_drift_right': os.path.join(DATASET_DIR, 'drift_right')
}

try:
    for key, path in dirs.items():
        if not os.path.exists(path):
            os.makedirs(path)
    print(f"Složky v '{DATASET_DIR}' byly zkontrolovány/vytvořeny.")
except Exception as e:
    print(f"Chyba při vytváření složek: {e}")

Složky v 'dataset_navigator_10class' byly zkontrolovány/vytvořeny.


In [3]:
import ipywidgets as widgets
from uuid import uuid1
import os

# --- VZHLED TLAČÍTEK ---
btn_layout = widgets.Layout(width='180px', height='60px')

# Funkce pro vytvoření tlačítka a počítadla
def create_control(label, style, dir_path):
    btn = widgets.Button(description=label, button_style=style, layout=btn_layout)
    # Pokud složka existuje, zjistíme počet, jinak 0
    count = len(os.listdir(dir_path)) if os.path.exists(dir_path) else 0
    cnt_widget = widgets.IntText(value=count, layout=widgets.Layout(width='80px', height='60px'))
    return btn, cnt_widget

# --- VYTVOŘENÍ OVLÁDACÍCH PRVKŮ ---

btn_dead, cnt_dead     = create_control('0: DEAD END (Slepá)', 'danger', dirs['0_dead_end'])
btn_corr, cnt_corr     = create_control('1: CORRIDOR (Rovně)', 'success', dirs['1_corridor'])
btn_corn_l, cnt_corn_l = create_control('2: CORNER LEFT', 'warning', dirs['2_corner_left'])
btn_corn_r, cnt_corn_r = create_control('3: CORNER RIGHT', 'warning', dirs['3_corner_right'])
btn_t_lr, cnt_t_lr     = create_control('4: T-JUNCT LR', 'info', dirs['4_t_junction_lr'])
btn_t_sl, cnt_t_sl     = create_control('5: T-JUNCT SL', 'primary', dirs['5_t_junction_sl'])
btn_t_sr, cnt_t_sr     = create_control('6: T-JUNCT SR', 'primary', dirs['6_t_junction_sr'])
btn_goal, cnt_goal     = create_control('7: GOAL (Cíl)', 'danger', dirs['7_goal'])
btn_drift_l, cnt_drift_l = create_control('8: DRIFT LEFT (Srovnej P)', 'info', dirs['8_drift_left'])
btn_drift_r, cnt_drift_r = create_control('9: DRIFT RIGHT (Srovnej L)', 'info', dirs['9_drift_right'])


# --- LOGIKA UKLÁDÁNÍ ---
def save_snapshot(folder_path, count_widget):
    image_path = os.path.join(folder_path, f"{uuid1()}.jpg")
    with open(image_path, 'wb') as f:
        f.write(image_widget.value)
    count_widget.value = len(os.listdir(folder_path))

# Lambda funkce pro kliknutí
btn_dead.on_click(lambda x: save_snapshot(dirs['0_dead_end'], cnt_dead))
btn_corr.on_click(lambda x: save_snapshot(dirs['1_corridor'], cnt_corr))
btn_corn_l.on_click(lambda x: save_snapshot(dirs['2_corner_left'], cnt_corn_l))
btn_corn_r.on_click(lambda x: save_snapshot(dirs['3_corner_right'], cnt_corn_r))
btn_t_lr.on_click(lambda x: save_snapshot(dirs['4_t_junction_lr'], cnt_t_lr))
btn_t_sl.on_click(lambda x: save_snapshot(dirs['5_t_junction_sl'], cnt_t_sl))
btn_t_sr.on_click(lambda x: save_snapshot(dirs['6_t_junction_sr'], cnt_t_sr))
btn_goal.on_click(lambda x: save_snapshot(dirs['7_goal'], cnt_goal))
# Nové
btn_drift_l.on_click(lambda x: save_snapshot(dirs['8_drift_left'], cnt_drift_l))
btn_drift_r.on_click(lambda x: save_snapshot(dirs['9_drift_right'], cnt_drift_r))


# --- ROZVRŽENÍ (LAYOUT) ---
# Sloupec VLEVO (Věci co zatáčí doleva nebo jsou vlevo)
col_left = widgets.VBox([
    widgets.HBox([btn_corn_l, cnt_corn_l]),   # 2
    widgets.HBox([btn_t_sl, cnt_t_sl]),       # 5
    widgets.HTML("<b>Korekce:</b>"),
    widgets.HBox([btn_drift_l, cnt_drift_l])  # 8 (Jsem vlevo)
])

# Sloupec STŘED
col_mid = widgets.VBox([
    widgets.HBox([btn_corr, cnt_corr]),       # 1
    widgets.HBox([btn_dead, cnt_dead]),       # 0
    widgets.HTML("<hr>"),
    widgets.HBox([btn_t_lr, cnt_t_lr]),       # 4
    widgets.HBox([btn_goal, cnt_goal])        # 7
])

# Sloupec VPRAVO (Věci co zatáčí doprava nebo jsou vpravo)
col_right = widgets.VBox([
    widgets.HBox([btn_corn_r, cnt_corn_r]),   # 3
    widgets.HBox([btn_t_sr, cnt_t_sr]),       # 6
    widgets.HTML("<b>Korekce:</b>"),
    widgets.HBox([btn_drift_r, cnt_drift_r]) # 9 (Jsem vpravo)
])

ui = widgets.HBox([col_left, col_mid, col_right])
display(image_widget)
display(ui)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [3]:
!zip -r -q dataset_maze.zip dataset_maze
print("Dataset zabalen. Můžete stahovat.")

Dataset zabalen. Můžete stahovat.
